<a href="https://colab.research.google.com/github/LP-D/claude/blob/main/notebooks/VIX_OHLC_VOL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# VIX OHLC Volatility Estimators v1 — nouvelle famille de features (6/6)

**Rôle.** Nouveau notebook indépendant, à lancer après `VIX_FINAL_FEATURES`. Teste une famille de
features **jamais essayée dans ce projet** : jusqu'ici, toutes les features de volatilité (Kalman,
HMM, EGARCH, Heston, VRP, Hawkes, Hurst/semivariance) sont construites à partir des seuls prix de
**clôture**. Ce notebook télécharge l'**OHLC** (Open/High/Low/Close) quotidien du VIX et du S&P 500
et construit des estimateurs de volatilité réalisée qui exploitent l'information intra-journalière
déjà présente dans une simple bougie journalière — sans données intraday, sans nouveaux tickers.

**Estimateurs** (tous causals, calculés sur fenêtres glissantes de 10/20/60 jours se terminant à
la date `t`) : **Parkinson** (extrême haut/bas), **Garman-Klass** (haut/bas + ouverture/clôture),
**Rogers-Satchell** (insensible à la dérive, donc plus adapté qu'GK en marché tendanciel),
**Yang-Zhang** (ajoute la variance du gap overnight open vs clôture veille — directement pertinent
pour capturer le risque de saut/spike déjà étudié via Hawkes et le filtre particulaire).

**Protocole de test** : A/B test façon `VIX_SPIKE_FEATURES` — pool de base vs pool de base + ces
features, sur GLOBAL/CALM/NORMAL/STRESS × 6 horizons × RandomForest/XGBoost, walk-forward,
N=8 fixe, sampler SMOTE fixe (pour isoler l'effet des seules nouvelles features). Le taux de
sélection SHAP de ces features est aussi tracké, comme dans `VIX_SPIKE_SCAN`.

**Comparateurs déjà établis** (walk-forward) : GLOBAL RandomForest h=5j (F1_dir≈0.610±0.025,
F1_UP_FORT≈0.359). Rappel du précédent le plus proche : dans `VIX_SPIKE_SCAN`, seules 2-3 features
sur ~15 candidates (ratio VIX/VIX3M, semivariance haussière) ont montré un taux de sélection
non-nul — ce notebook vérifie si les estimateurs OHLC, orthogonaux à ces familles, font mieux.


In [ ]:
import subprocess, sys
pkgs = ['xgboost', 'shap', 'xlsxwriter', 'imbalanced-learn', 'pyarrow', 'yfinance']
subprocess.run([sys.executable,'-m','pip','install','-q']+pkgs, check=False)
print("Installation OK")


In [ ]:
import os, time, json, warnings, random
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import yfinance as yf
import shap
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import f1_score, accuracy_score
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE

SEED = 42; random.seed(SEED); np.random.seed(SEED)

NOTEBOOK_NAME = 'VIX_OHLC_VOL'
NOTEBOOK_VERSION = 'v1'

CONFIG = {
    'flat_thr': 0.003,
    'horizons': [1, 2, 3, 5, 7, 10],
    'regimes': ['GLOBAL', 'CALM', 'NORMAL', 'STRESS'],
    'algos': ['RandomForest', 'XGBoost'],
    'windows': [10, 20, 60],
    'ohlc_tickers': {'VIX': '^VIX', 'SPX': '^GSPC'},
    'N': 8,               # point de fonctionnement établi
    'sampler': 'SMOTE',
    'n_wf_folds': 5,
    'min_train_frac': 0.40,  # doit matcher VIX_FINAL_FEATURES
    'shap_sample': 500,
    'pool_prefilter': 450,
    'min_train_rows': 100, 'min_test_rows': 20,
}
TARGET_COL = 'VIX_Amplitude_Class'
RESULTS_CSV = 'vix_ohlc_vol_results.csv'
GITHUB_REPO = 'LP-D/claude'
FEATURES_BRANCH = 'results/vix-final-features'
RESULTS_BRANCH = 'results/vix-ohlc-vol'

n_combos = (len(CONFIG['horizons']) * len(CONFIG['regimes']) * len(CONFIG['algos'])
            * CONFIG['n_wf_folds'] * 2)
print(f"{NOTEBOOK_NAME} {NOTEBOOK_VERSION} | N={CONFIG['N']} sampler={CONFIG['sampler']} | "
      f"grille = {n_combos} lignes ({len(CONFIG['horizons'])}h × {len(CONFIG['regimes'])}reg × "
      f"{len(CONFIG['algos'])}algos × {CONFIG['n_wf_folds']}folds × 2 [pool BASE / BASE+OHLC])")


In [ ]:
# ============================================================
# CHARGEMENT DU DATASET PARTAGÉ (produit par VIX_FINAL_FEATURES)
# ============================================================
import subprocess

try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN')

if not os.path.exists('vix_final_features.parquet'):
    auth = f"{GITHUB_TOKEN}@" if GITHUB_TOKEN else ""
    url = f"https://{auth}github.com/{GITHUB_REPO}.git"
    workdir = "/content/_vix_features_pull"
    subprocess.run(["rm", "-rf", workdir], check=False)
    clone = subprocess.run(["git", "clone", "--depth", "1", "--branch", FEATURES_BRANCH, url, workdir],
                           capture_output=True, text=True)
    if clone.returncode != 0:
        raise RuntimeError(
            "Impossible de récupérer le dataset partagé depuis "
            f"'{FEATURES_BRANCH}'. As-tu bien exécuté VIX_FINAL_FEATURES.ipynb en premier "
            f"(et poussé son résultat) ? Détail: {clone.stderr[-500:]}")
    subprocess.run(["cp", f"{workdir}/vix_final_features.parquet", "."], check=True)
    subprocess.run(["cp", f"{workdir}/vix_final_features_meta.json", "."], check=True)
    print(f"[PULL OK] Dataset récupéré depuis '{FEATURES_BRANCH}'")
else:
    print("[SKIP] vix_final_features.parquet déjà présent localement")

df_features = pd.read_parquet('vix_final_features.parquet')
with open('vix_final_features_meta.json') as f:
    meta = json.load(f)
FEATURE_POOL_BASE = meta['feature_pool']
VIX_COL = meta['vix_col']; SPX_COL = meta['spx_col']
print(f"Dataset: {df_features.shape} | VIX={VIX_COL} | pool de base: {len(FEATURE_POOL_BASE)} features "
      f"| source: {meta['date_min']} → {meta['date_max']}")

all_dates = df_features.dropna(how='all').index.sort_values()
n_obs = len(all_dates)
first_cut = int(n_obs * CONFIG['min_train_frac'])
test_span = (n_obs - first_cut) // CONFIG['n_wf_folds']
FOLD_CUTS = [first_cut + k * test_span for k in range(CONFIG['n_wf_folds'] + 1)]
FOLD_CUTS[-1] = n_obs
for k in range(CONFIG['n_wf_folds']):
    print(f"  Fold {k+1}: train → {all_dates[FOLD_CUTS[k]-1].date()} | "
          f"test {all_dates[FOLD_CUTS[k]].date()} → {all_dates[FOLD_CUTS[k+1]-1].date()}")


In [ ]:
# ============================================================
# TÉLÉCHARGEMENT OHLC (VIX + SPX) — robuste, un ticker à la fois
# ============================================================
def download_ohlc_robust(ticker, start='1990-01-01'):
    for attempt in range(3):
        try:
            df = yf.download(ticker, start=start, progress=False, auto_adjust=False)
            if df is not None and len(df) > 0:
                if isinstance(df.columns, pd.MultiIndex):
                    df.columns = df.columns.get_level_values(0)
                out = df[['Open', 'High', 'Low', 'Close']].copy()
                out.index = pd.to_datetime(out.index).tz_localize(None)
                return out
        except Exception as e:
            print(f"  [WARN] {ticker} tentative {attempt+1}: {e}")
    print(f"  [ECHEC] {ticker} — pas de données OHLC après 3 tentatives.")
    return None

ohlc = {}
for name, ticker in CONFIG['ohlc_tickers'].items():
    df_o = download_ohlc_robust(ticker)
    if df_o is not None:
        ohlc[name] = df_o.reindex(all_dates).ffill()
        print(f"  [OK] {name} ({ticker}): {df_o.shape[0]} lignes brutes → réindexé sur {len(all_dates)} dates")
    else:
        ohlc[name] = None

if all(v is None for v in ohlc.values()):
    raise RuntimeError("Aucun OHLC récupéré (VIX et SPX) — impossible de construire les features de ce notebook.")


In [ ]:
# ============================================================
# CONSTRUCTION DES ESTIMATEURS DE VOLATILITÉ RÉALISÉE (OHLC), CAUSALS
# ============================================================
for name, df_o in ohlc.items():
    if df_o is None:
        continue
    O, H, L, C = df_o['Open'], df_o['High'], df_o['Low'], df_o['Close']
    C_prev = C.shift(1)
    log_hl = np.log(H / L)
    log_co = np.log(C / O)
    log_hc = np.log(H / C)
    log_ho = np.log(H / O)
    log_lc = np.log(L / C)
    log_lo = np.log(L / O)
    log_oc_prev = np.log(O / C_prev)
    rs_daily = log_hc * log_ho + log_lc * log_lo

    for w in CONFIG['windows']:
        park = np.sqrt((log_hl ** 2).rolling(w).mean() / (4 * np.log(2))) * np.sqrt(252)
        gk_var = (0.5 * (log_hl ** 2) - (2 * np.log(2) - 1) * (log_co ** 2)).rolling(w).mean()
        gk = np.sqrt(gk_var.clip(lower=0)) * np.sqrt(252)
        rs = np.sqrt(rs_daily.rolling(w).mean().clip(lower=0)) * np.sqrt(252)
        k = 0.34 / (1.34 + (w + 1) / (w - 1))
        var_o = (log_oc_prev ** 2).rolling(w).mean()
        var_c = (log_co ** 2).rolling(w).mean()
        var_rs = rs_daily.rolling(w).mean()
        yz = np.sqrt((var_o + k * var_c + (1 - k) * var_rs).clip(lower=0)) * np.sqrt(252)

        df_features[f'ohlc_{name.lower()}_parkinson_{w}d'] = park
        df_features[f'ohlc_{name.lower()}_gk_{w}d'] = gk
        df_features[f'ohlc_{name.lower()}_rs_{w}d'] = rs
        df_features[f'ohlc_{name.lower()}_yz_{w}d'] = yz

OHLC_FEATURES = [c for c in df_features.columns if c.startswith('ohlc_')]
FEATURE_POOL_PLUS = FEATURE_POOL_BASE + OHLC_FEATURES
print(f"{len(OHLC_FEATURES)} nouvelles features OHLC construites (Parkinson/GK/RS/YZ × "
      f"{len(CONFIG['windows'])} fenêtres × {sum(v is not None for v in ohlc.values())} ticker(s) dispo)")


## Principe : estimateurs de volatilité réalisée à partir de l'OHLC

Une bougie journalière (Open/High/Low/Close) contient plus d'information sur la volatilité
intra-journalière qu'un simple rendement close-to-close — sans nécessiter de données intraday.

- **Parkinson (1980)** : $\sigma_P^2 = \frac{1}{4\ln 2}\, \overline{\left[\ln(H_t/L_t)\right]^2}$
  — utilise l'amplitude extrême haut/bas. Plus efficace que close-to-close si pas de gap
  d'ouverture, mais ignore le drift et les gaps overnight.

- **Garman-Klass (1980)** : $\sigma_{GK}^2 = \overline{0.5\left[\ln(H_t/L_t)\right]^2 - (2\ln 2 - 1)\left[\ln(C_t/O_t)\right]^2}$
  — ajoute l'information ouverture/clôture, meilleure efficacité statistique que Parkinson pour
  un même nombre d'observations, mais suppose un processus sans drift ni saut overnight.

- **Rogers-Satchell (1991)** : $\sigma_{RS}^2 = \overline{\ln(H_t/C_t)\ln(H_t/O_t) + \ln(L_t/C_t)\ln(L_t/O_t)}$
  — seul estimateur classique **indépendant de la dérive** (le VIX a des tendances marquées en
  période de stress), donc potentiellement plus fiable que GK/Parkinson dans les régimes STRESS.

- **Yang-Zhang (2000)** : $\sigma_{YZ}^2 = \sigma_{overnight}^2 + k\,\sigma_{open\text{-}close}^2 + (1-k)\,\sigma_{RS}^2$
  où $\sigma_{overnight}^2$ est la variance de $\ln(O_t/C_{t-1})$ (le **gap** veille→ouverture) et
  $k = \frac{0.34}{1.34 + \frac{n+1}{n-1}}$. C'est l'estimateur le plus complet : il capture
  explicitement le risque de **saut overnight**, directement lié à la thématique des sauts déjà
  étudiée dans ce projet via le processus de Hawkes et le filtre particulaire — mais jamais sous
  cet angle "réalisé à partir de l'OHLC" jusqu'ici.

Toutes ces variances sont annualisées ($\times\sqrt{252}$ sur l'écart-type) et calculées sur des
fenêtres glissantes **se terminant strictement à la date `t`** (donc causales), pour le VIX et le
S&P 500, sur 10/20/60 jours.


In [ ]:
def build_target(vix_series, horizon, split_idx):
    vix = vix_series.ffill().bfill(); vix_tr = vix.iloc[:split_idx]
    calm_thr = vix_tr.quantile(0.33); stress_thr = vix_tr.quantile(0.67)
    regime = pd.Series('NORMAL', index=vix.index)
    regime[vix < calm_thr] = 'CALM'; regime[vix >= stress_thr] = 'STRESS'
    ret = (vix.shift(-horizon) / vix) - 1
    flat = ret.abs() < CONFIG['flat_thr']
    ret = ret.loc[~flat].dropna(); reg_r = regime.reindex(ret.index)
    cut_date = vix.index[min(split_idx, len(vix) - 1)]
    ret_tr = ret.loc[ret.index < cut_date]; reg_tr = reg_r.loc[ret_tr.index]
    thr = {}
    for reg in ['CALM', 'NORMAL', 'STRESS']:
        sub = ret_tr[reg_tr == reg]
        thr[reg] = (sub.quantile(0.25) if len(sub) >= 20 else ret_tr.quantile(0.25),
                    sub.quantile(0.75) if len(sub) >= 20 else ret_tr.quantile(0.75))
    thr['GLOBAL'] = (ret_tr.quantile(0.25), ret_tr.quantile(0.75))
    def classify(r, reg):
        q25, q75 = thr.get(reg, (0, 0))
        if r < q25: return 0
        if r < 0:   return 1
        if r < q75: return 2
        return 3
    target = pd.Series([classify(r, reg_r[i]) for i, r in ret.items()], index=ret.index, name=TARGET_COL)
    return target, reg_r, thr

def metrics(y_true, y_pred):
    dm = {0: 'DOWN', 1: 'DOWN', 2: 'UP', 3: 'UP'}
    yd_t = [dm[y] for y in y_true]; yd_p = [dm[y] for y in y_pred]
    m = {'F1_4cls': round(f1_score(y_true, y_pred, average='macro', zero_division=0), 4),
         'Acc_dir': round(accuracy_score(yd_t, yd_p), 4),
         'F1_dir': round(f1_score(yd_t, yd_p, average='macro', zero_division=0), 4)}
    ui = [i for i, y in enumerate(y_true) if dm[y] == 'UP']
    di = [i for i, y in enumerate(y_true) if dm[y] == 'DOWN']
    if len(ui) >= 10:
        yt = ['FORT' if y_true[i] == 3 else 'FAIBLE' for i in ui]
        yp = ['FORT' if y_pred[i] == 3 else 'FAIBLE' for i in ui]
        m['F1_UP_FORT'] = round(f1_score(yt, yp, pos_label='FORT', average='binary', zero_division=0), 4)
    else: m['F1_UP_FORT'] = np.nan
    if len(di) >= 10:
        yt = ['FORT' if y_true[i] == 0 else 'FAIBLE' for i in di]
        yp = ['FORT' if y_pred[i] == 0 else 'FAIBLE' for i in di]
        m['F1_DOWN_FORT'] = round(f1_score(yt, yp, pos_label='FORT', average='binary', zero_division=0), 4)
    else: m['F1_DOWN_FORT'] = np.nan
    return m

def get_clf(algo):
    if algo == 'XGBoost':
        return XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05, subsample=0.8,
                             colsample_bytree=0.8, min_child_weight=3, eval_metric='mlogloss',
                             objective='multi:softprob', random_state=SEED, n_jobs=-1, verbosity=0)
    if algo == 'RandomForest':
        return RandomForestClassifier(n_estimators=200, max_depth=6, min_samples_leaf=5,
                                      class_weight='balanced', random_state=SEED, n_jobs=-1)
    raise ValueError(algo)

def get_samp(name):
    return {'SMOTE': SMOTE(random_state=SEED)}[name]

def shap_rank(X_tr, y_tr, pool_names, top_n, prefilter):
    nf = X_tr.shape[1]
    if nf > prefilter:
        pf = XGBClassifier(n_estimators=60, max_depth=4, learning_rate=0.1, objective='multi:softprob',
                           eval_metric='mlogloss', random_state=SEED, n_jobs=-1, verbosity=0)
        pf.fit(X_tr, y_tr); keep = np.argsort(pf.feature_importances_)[::-1][:prefilter]
    else:
        keep = np.arange(nf)
    Xk = X_tr[:, keep]
    pilot = XGBClassifier(n_estimators=80, max_depth=4, learning_rate=0.1, objective='multi:softprob',
                          eval_metric='mlogloss', random_state=SEED, n_jobs=-1, verbosity=0)
    pilot.fit(Xk, y_tr)
    sv = np.abs(np.array(shap.TreeExplainer(pilot).shap_values(Xk[:min(CONFIG['shap_sample'], len(Xk))])))
    nfk = Xk.shape[1]
    feat_axes = [ax for ax in range(sv.ndim) if sv.shape[ax] == nfk]
    if len(feat_axes) == 1:
        arr = sv.mean(axis=tuple(ax for ax in range(sv.ndim) if ax != feat_axes[0]))
    else:
        arr = np.asarray(pilot.feature_importances_)
    order = np.argsort(np.asarray(arr).ravel())[::-1][:top_n]
    return list(keep[order])

print("Helpers OK (build_target, metrics, get_clf, get_samp, shap_rank)")


In [ ]:
# ============================================================
# SYNCHRONISATION DE LA PROGRESSION (même principe que les autres notebooks)
# ============================================================
_PUSH_WORKDIR = "/content/_vix_ohlc_push"

def push_progress(label=''):
    if not GITHUB_TOKEN or not os.path.exists(RESULTS_CSV):
        return False
    try:
        subprocess.run(["rm", "-rf", _PUSH_WORKDIR], check=False)
        url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO}.git"
        clone = subprocess.run(["git", "clone", url, _PUSH_WORKDIR], capture_output=True, text=True)
        if clone.returncode != 0: return False
        exists = subprocess.run(["git", "-C", _PUSH_WORKDIR, "ls-remote", "--exit-code", "--heads",
                                  "origin", RESULTS_BRANCH], capture_output=True, text=True)
        if exists.returncode == 0:
            subprocess.run(["git", "-C", _PUSH_WORKDIR, "checkout", "-B", RESULTS_BRANCH,
                             f"origin/{RESULTS_BRANCH}"], check=True)
        else:
            subprocess.run(["git", "-C", _PUSH_WORKDIR, "checkout", "-B", RESULTS_BRANCH], check=True)
        subprocess.run(["cp", RESULTS_CSV, f"{_PUSH_WORKDIR}/{RESULTS_CSV}"], check=True)
        subprocess.run(["git", "-C", _PUSH_WORKDIR, "config", "user.email", "vix-colab@users.noreply.github.com"], check=True)
        subprocess.run(["git", "-C", _PUSH_WORKDIR, "config", "user.name", "VIX OHLC Vol Colab run"], check=True)
        subprocess.run(["git", "-C", _PUSH_WORKDIR, "add", RESULTS_CSV], check=True)
        subprocess.run(["git", "-C", _PUSH_WORKDIR, "commit", "-m",
                       f"Progression OHLC Vol {label} — {pd.Timestamp.now():%Y-%m-%d %H:%M}"],
                       capture_output=True, text=True)
        push = subprocess.run(["git", "-C", _PUSH_WORKDIR, "push", "origin", RESULTS_BRANCH],
                              capture_output=True, text=True)
        ok = push.returncode == 0
        if ok: print(f"  [CHECKPOINT PUSHÉ] {label}")
        return ok
    except Exception as e:
        print(f"  [WARN push checkpoint] {e}")
        return False

def pull_progress():
    if os.path.exists(RESULTS_CSV) or not GITHUB_TOKEN:
        return
    url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO}.git"
    exists = subprocess.run(["git", "ls-remote", "--exit-code", "--heads", url, RESULTS_BRANCH],
                            capture_output=True, text=True)
    if exists.returncode != 0:
        print(f"[INFO] Aucune progression antérieure sur '{RESULTS_BRANCH}'.")
        return
    workdir = "/content/_vix_ohlc_pull"
    subprocess.run(["rm", "-rf", workdir], check=False)
    clone = subprocess.run(["git", "clone", "--depth", "1", "--branch", RESULTS_BRANCH, url, workdir],
                           capture_output=True, text=True)
    if clone.returncode == 0 and os.path.exists(f"{workdir}/{RESULTS_CSV}"):
        subprocess.run(["cp", f"{workdir}/{RESULTS_CSV}", "."], check=True)
        print("[PULL OK] Progression OHLC Vol antérieure récupérée.")

pull_progress()


In [ ]:
# ============================================================
# A/B : POOL DE BASE vs POOL DE BASE + ESTIMATEURS OHLC
# ============================================================
KEY_COLS = ['horizon', 'regime', 'algo', 'fold', 'pool_variant']
OHLC_SET = set(OHLC_FEATURES)

done_keys = set()
if os.path.exists(RESULTS_CSV) and os.path.getsize(RESULTS_CSV) > 0:
    prev = pd.read_csv(RESULTS_CSV, usecols=KEY_COLS)
    done_keys = set(map(tuple, prev.values.tolist()))
    print(f"[REPRISE] {len(done_keys)} lignes déjà calculées.")

def save_row(row):
    header = not (os.path.exists(RESULTS_CSV) and os.path.getsize(RESULTS_CSV) > 0)
    pd.DataFrame([row]).to_csv(RESULTS_CSV, mode='a', header=header, index=False)
    done_keys.add(tuple(row[c] for c in KEY_COLS))

POOLS = {'BASE': FEATURE_POOL_BASE, 'BASE_PLUS_OHLC': FEATURE_POOL_PLUS}

t0 = time.time()
n_saved = 0
for h in CONFIG['horizons']:
    for reg in CONFIG['regimes']:
        for k in range(CONFIG['n_wf_folds']):
            cut, nxt = FOLD_CUTS[k], FOLD_CUTS[k + 1]
            cut_date, nxt_date = all_dates[cut], all_dates[nxt - 1]
            target, reg_r, _ = build_target(df_features[VIX_COL], h, cut)
            idx = target.index
            tr_mask = np.asarray(idx < cut_date)
            te_mask = np.asarray((idx >= cut_date) & (idx <= nxt_date))
            if reg != 'GLOBAL':
                reg_al = reg_r.reindex(idx).fillna('NORMAL').values
                tr_mask = tr_mask & (reg_al == reg); te_mask = te_mask & (reg_al == reg)
            y_tr = target.values[tr_mask].astype(int); y_te = target.values[te_mask].astype(int)
            if len(y_tr) < CONFIG['min_train_rows'] or len(y_te) < CONFIG['min_test_rows']:
                continue

            for algo in CONFIG['algos']:
                for variant, pool in POOLS.items():
                    key = (h, reg, algo, k, variant)
                    if key in done_keys:
                        continue
                    X_pool = df_features[pool].reindex(idx)
                    sc = RobustScaler()
                    X_tr = sc.fit_transform(np.nan_to_num(X_pool.values[tr_mask]))
                    X_te = sc.transform(np.nan_to_num(X_pool.values[te_mask]))
                    fidx = shap_rank(X_tr, y_tr, pool, CONFIG['N'], CONFIG['pool_prefilter'])
                    selected = [pool[i] for i in fidx]
                    n_ohlc_selected = sum(1 for f in selected if f in OHLC_SET)
                    try:
                        Xr, yr = get_samp(CONFIG['sampler']).fit_resample(X_tr[:, fidx], y_tr)
                    except Exception:
                        Xr, yr = X_tr[:, fidx], y_tr
                    clf = get_clf(algo); clf.fit(Xr, yr)
                    met = metrics(y_te, clf.predict(X_te[:, fidx]))
                    save_row({'horizon': h, 'regime': reg, 'algo': algo, 'fold': k, 'pool_variant': variant,
                              'n_train': len(y_tr), 'n_test': len(y_te),
                              'n_ohlc_selected': n_ohlc_selected if variant == 'BASE_PLUS_OHLC' else 0,
                              'selected_features': json.dumps(selected), **met})
                    n_saved += 1
                    if n_saved % 200 == 0:
                        print(f"  ... {len(done_keys)}/{n_combos} lignes | {(time.time()-t0)/60:.1f}min")
                        push_progress(label=f"{len(done_keys)}/{n_combos}")

print(f"\n[OHLC VOL] {len(done_keys)}/{n_combos} lignes calculées ({(time.time()-t0)/60:.1f}min)")
push_progress(label='fin de run')


In [ ]:
# ============================================================
# SYNTHÈSE : APPORT DES ESTIMATEURS OHLC
# ============================================================
df_o = pd.read_csv(RESULTS_CSV) if os.path.exists(RESULTS_CSV) else pd.DataFrame()
print(f"Progression: {len(df_o)}/{n_combos} ({len(df_o)/max(n_combos,1):.1%})")

if len(df_o):
    ok = df_o.dropna(subset=['F1_dir'])
    piv = ok.pivot_table(index=['horizon', 'regime', 'algo'], columns='pool_variant',
                          values=['F1_dir', 'F1_UP_FORT', 'F1_DOWN_FORT'], aggfunc='mean')
    if ('BASE' in piv['F1_dir'].columns) and ('BASE_PLUS_OHLC' in piv['F1_dir'].columns):
        delta = pd.DataFrame({
            'F1_dir_BASE': piv['F1_dir']['BASE'].round(4),
            'F1_dir_BASE_PLUS_OHLC': piv['F1_dir']['BASE_PLUS_OHLC'].round(4),
            'delta_F1_dir': (piv['F1_dir']['BASE_PLUS_OHLC'] - piv['F1_dir']['BASE']).round(4),
            'F1_UP_FORT_BASE': piv['F1_UP_FORT']['BASE'].round(4),
            'F1_UP_FORT_BASE_PLUS_OHLC': piv['F1_UP_FORT']['BASE_PLUS_OHLC'].round(4),
            'delta_F1_UP_FORT': (piv['F1_UP_FORT']['BASE_PLUS_OHLC'] - piv['F1_UP_FORT']['BASE']).round(4),
        }).reset_index().sort_values('delta_F1_dir', ascending=False)
        print("\n### Delta (BASE+OHLC - BASE) par (horizon, régime, algo) ###")
        print(delta.to_string(index=False))
        print(f"\nDelta F1_dir moyen : {delta['delta_F1_dir'].mean():+.4f} "
              f"(std {delta['delta_F1_dir'].std():.4f})")
        print(f"Delta F1_UP_FORT moyen : {delta['delta_F1_UP_FORT'].mean():+.4f}")

        plus = ok[ok['pool_variant'] == 'BASE_PLUS_OHLC']
        sel_rate = (plus['n_ohlc_selected'] > 0).mean()
        print(f"\nTaux de lignes où ≥1 feature OHLC est sélectionnée par SHAP : {sel_rate:.1%} "
              f"({len(plus)} configs testées)")
        from collections import Counter
        cnt = Counter()
        for s in plus['selected_features']:
            try:
                for f in json.loads(s):
                    if f in OHLC_SET: cnt[f] += 1
            except Exception:
                pass
        if cnt:
            print("\nFeatures OHLC les plus sélectionnées :")
            for f, c in cnt.most_common(10):
                print(f"  {f}: {c}/{len(plus)} ({c/len(plus):.1%})")

        if delta['delta_F1_dir'].mean() > 0.01 or sel_rate > 0.15:
            print("\n[VERDICT] Signal positif — les estimateurs OHLC apportent quelque chose et "
                  "méritent d'être ajoutés au pool de features partagé (VIX_FINAL_FEATURES).")
        else:
            print("\n[VERDICT] Pas de signal probant — comme les features 'spike' précédentes, "
                  "cette famille orthogonale n'apporte pas d'amélioration matérielle sur "
                  "F1_dir/F1_UP_FORT en walk-forward.")

        try:
            with pd.ExcelWriter('VIX_OHLC_VOL_report.xlsx', engine='xlsxwriter') as w:
                delta.to_excel(w, 'Delta_BASE_vs_OHLC', index=False)
                df_o.drop(columns=['selected_features']).to_excel(w, 'Detail', index=False)
            print("\n[SAVE] VIX_OHLC_VOL_report.xlsx (snapshot à date)")
        except Exception as e:
            print(f"[WARN Export] {e}")
    else:
        print("Pas encore assez de lignes des deux côtés (BASE / BASE_PLUS_OHLC) pour comparer.")
else:
    print("Aucun résultat pour l'instant.")
print(f"\n[NOTE] {RESULTS_CSV} contient le détail complet — le recharger pour reprendre.")


In [ ]:
# ============================================================
# PUSH FINAL DU RAPPORT (xlsx) EN PLUS DU CSV DE PROGRESSION
# ============================================================
def push_report_file():
    if not GITHUB_TOKEN or not os.path.exists('VIX_OHLC_VOL_report.xlsx'):
        print("[SKIP] Pas de token ou pas de rapport à pousser.")
        return
    try:
        subprocess.run(["cp", "VIX_OHLC_VOL_report.xlsx", f"{_PUSH_WORKDIR}/VIX_OHLC_VOL_report.xlsx"], check=True)
        subprocess.run(["git", "-C", _PUSH_WORKDIR, "add", "VIX_OHLC_VOL_report.xlsx"], check=True)
        subprocess.run(["git", "-C", _PUSH_WORKDIR, "commit", "-m",
                       f"Rapport OHLC Vol agrégé — {pd.Timestamp.now():%Y-%m-%d %H:%M}"],
                       capture_output=True, text=True)
        push = subprocess.run(["git", "-C", _PUSH_WORKDIR, "push", "origin", RESULTS_BRANCH],
                              capture_output=True, text=True)
        if push.returncode == 0:
            print(f"[PUSH OK] VIX_OHLC_VOL_report.xlsx sur '{RESULTS_BRANCH}'")
        else:
            print(f"[WARN] {push.stderr[-300:]}")
    except Exception as e:
        print(f"[WARN] {e}")

push_progress(label='rapport final')
push_report_file()
